# 비유의성 원인 진단 보고서 생성

04 정밀도·05 노출·06 표본 진단과 기존 고정 회귀결과를 종합한다. 대상은 졸업프로젝트 연구자와 지도교수다. 보고서는 p값을 작게 만드는 사양 선택이 아니라, 검증된 측정 구조와 통계적 불확실성으로 연구 설계를 진단한다. HTML 보고서에 정의·증거·해석·한계·후속 질문을 함께 배치한다.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import numpy as np
import json, hashlib, subprocess, shutil
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'data/raw').is_dir())
D=ROOT/'results/main/null_diagnostics'
OUT=D
READABLE=ROOT/'reports/main/null_diagnostics'
READABLE.mkdir(parents=True,exist_ok=True)
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
load=lambda p:json.loads(p.read_text(encoding='utf-8'))
paths={'coverage':D/'sample_industry_coverage.csv','stores':D/'sample_store_coverage.csv',
       'selection':D/'sample_exposure_coverage.csv','ratio':D/'sample_night_amount_vs_share.csv',
       'information':ROOT/'results/main/baseline/09_information_concentration.csv',
       'primary':ROOT/'results/main/baseline/09_industry_effects.csv',
       'time_common':ROOT/'results/main/time_windows/11_common_effects.csv',
       'time_industry':ROOT/'results/main/time_windows/11_industry_effects.csv'}
frames={k:pd.read_csv(v) for k,v in paths.items()}
sample=load(D/'sample_summary.json')
old_plan=load(ROOT/'results/main/time_windows/11_analysis_summary.json')['plan']
assert old_plan['exact_event_window_sales'] is False
input_hashes={v.relative_to(ROOT).as_posix():sha(v) for v in paths.values()}
primary=frames['primary'].query("scenario=='main_500'").copy()
assert len(primary)==20
print('주분석 검정:',len(primary),'시간창 업종별 검정:',len(frames['time_industry']))

주분석 검정: 20 시간창 업종별 검정: 40


## 1. 정확한 표와 그래프용 데이터

그래프는 규모별·업종별 표본 제공률과 식별에 남은 변동을 비교한다. 표는 개별 신뢰구간, 표본수, 보정 전후 p값을 정확히 조회하는 용도로 사용한다. 모든 원 수치를 보존한다.

In [2]:
import sqlite3
connection=sqlite3.connect(":memory:")
datasets={}
sources=[]
def dataset(key,frame,path,description):
    frame.to_sql(key,connection,index=False,if_exists='replace')
    query='SELECT * FROM "'+key+'"'
    result_frame=pd.read_sql_query(query,connection)
    datasets[key]=json.loads(result_frame.to_json(orient='records',force_ascii=False))
    sources.append({'id':key,'label':description,'path':path.relative_to(ROOT).as_posix(),
        'query':{'engine':'sqlite','language':'sql','sql':query,'description':description,
                 'tables_used':[key],
                 'files_used':[path.relative_to(ROOT).as_posix(),'notebooks/main/reporting/07_비유의성_진단보고서.ipynb']}})
dataset('coverage',frames['coverage'].sort_values('positive_store_coverage_pct',ascending=False),paths['coverage'],
        '전체점포 수가 양수인 상권×업종×분기의 매출 관측 수 / 해당 집단 행수 ×100. 원격 자료가 아닌 고정 패널 기술통계.')
info=frames['information'].query("outcome=='log_sales'").copy()
info['top5_pct']=100*info.top5_information_share
dataset('information',info,paths['information'],'주모형에서 상권별 잔차화 노출 제곱합의 비중과 집중도. 역 Herfindahl은 정보분포 지수이며 검정 자유도가 아님.')
common=frames['time_common'].copy()
for df in [common]:
    df['estimate_ci']=df.apply(lambda r:f"{r.doubling_pct:+.2f}% [{r.doubling_ci_low:+.2f}%, {r.doubling_ci_high:+.2f}%]",axis=1)
    df['raw_p']=df.p_value.map(lambda x:f'{x:.4f}')
    df['adjusted_p']=df.p_holm_4.map(lambda x:f'{x:.4f}')
dataset('time_common',common,paths['time_common'],'ln(1+노출) 기울기×ln2를 지수변환한 (매출+1) 조건부 기하평균 상대차이와 개별95%CI. 시간대 상대패턴 비교.')
families=pd.DataFrame([
    {'대상':'주분석: 총매출·야간비중 ×10업종','검정수':len(primary),'보정전_5pct':int(primary.p_value.lt(.05).sum()),
     '보정후_5pct':int(primary.p_fdr_primary.lt(.05).sum()),'보정':'BH /20'},
    {'대상':'시간창: 4창 ×10업종','검정수':len(frames['time_industry']),'보정전_5pct':int(frames['time_industry'].p_value.lt(.05).sum()),
     '보정후_5pct':int(frames['time_industry'].p_bh_40.lt(.05).sum()),'보정':'BH /40'},
    {'대상':'시간창: 공통 기울기 4창','검정수':len(common),'보정전_5pct':int(common.p_value.lt(.05).sum()),
     '보정후_5pct':int(common.p_holm_4.lt(.05).sum()),'보정':'Holm /4'}])
family_path=D/'report_test_families.csv'
families.to_csv(family_path,index=False,encoding='utf-8-sig')
dataset('families',families,family_path,'기존 결과의 전체 검정군별 개수. 새로운 가설을 추가하거나 유의한 사양만 선택하지 않음.')


In [3]:
for key,name in [('variation','inference_identifying_variation.csv'),('contrasts','inference_contrasts.csv')]:
    paths[key]=D/name
    frames[key]=pd.read_csv(paths[key])
    input_hashes[paths[key].relative_to(ROOT).as_posix()]=sha(paths[key])
variance=frames['variation'].query("family=='primary20' and outcome=='log_sales'").copy()
assert len(variance)==10
variance=variance.sort_values('fe_variance_retained_pct',ascending=False)
dataset('variation',variance,paths['variation'],'업종별 실제 주모형 표본에서 중심화한 ln(1+공연 ID 수) 제곱합 대비 상권·분기 FE 제거 후 잔차 제곱합. 시간대 모형과 분모를 섞지 않음.')
ct=frames['contrasts'].query("family=='primary20' and contrast=='one_plus_exposure_doubled'").copy()
ct['effect_ci']=ct.apply(lambda r:f"{r.estimate:+.2f} [{r.ci_low:+.2f}, {r.ci_high:+.2f}]",axis=1)
precision=ct.pivot(index=['industry_id','industry'],columns='outcome',values='effect_ci').reset_index()
dataset('precision',precision,paths['contrasts'],'1+등록 공연 ID 수 2배의 모형상 비교. 총매출은 (매출+1) 조건부 기하평균 상대차이 %, 야간 비중은 %p. 개별95%구간이며 동시보정 구간이 아님.')


In [4]:
exposure=load(D/'exposure_summary.json')
inference=load(D/'inference_summary.json')
assert exposure['all_inputs_unchanged'] and inference['protected_unchanged']
assert inference['reproduced_coefficients']==64
for key,name in [('sample_summary','sample_summary.json'),('exposure_summary','exposure_summary.json'),('inference_summary','inference_summary.json')]:
    path=D/name
    sources.append({'id':key,'label':name,'path':path.relative_to(ROOT).as_posix()})
    input_hashes[path.relative_to(ROOT).as_posix()]=sha(path)


## 2. 보고서 본문과 근거 배치

요약 → 측정 대상 → 정밀도 → 표본·노출 구조 → 지표 해석 → 원인별 증거 강도 → 후속 설계 순서다. 시각화는 비교용 막대그래프이고 인과적 기여도 분해로 제시하지 않는다.

In [5]:
title='공연·외식업 분석에서 유의한 관계가 뚜렷하지 않은 이유'
blocks=[];tables=[];charts=[]
def md(key,body,source=None):
    block={'id':key,'type':'markdown','body':body}
    if source: block['sourceId']=source
    blocks.append(block)
def table(key,title,columns,sort):
    tables.append({'id':key,'title':title,'dataset':key,'sourceId':key,
       'defaultSort':{'field':sort,'direction':'asc'},
       'columns':[{'field':f,'label':label,'type':typ,**({'format':'number'} if typ=='number' else {})} for f,label,typ in columns]})
    blocks.append({'id':key+'_block','type':'table','tableId':key})
def chart(key,title,x,y,label,subtitle):
    charts.append({'id':key+'_chart','title':title,'dataset':key,'sourceId':key,'type':'horizontalBar',
      'subtitle':subtitle,'showDescription':True,'valueFormat':'number','unit':'%',
      'encodings':{'x':{'field':x,'type':'nominal','label':'업종'},
                   'y':{'field':y,'type':'quantitative','label':label}}})
    blocks.append({'id':key+'_chart_block','type':'chart','chartId':key+'_chart'})
md('title','# '+title+'\n\n**졸업 프로젝트 · 기존 분석의 원인 진단**  \n종로구·마포구 · 외식업 10종 · 2023~2025년 · 진단일 2026년 10월 5일')
md('summary','## 요약: 측정 대상의 차이와 제한된 비교 정보가 함께 있다\n\n현재 결과를 단순히 “공연은 매출에 영향을 주지 않는다”로 결론내릴 수 없다. 공연의 실제 관객 유입과 소비를 직접 관측하지 못하며, 작품 수·예정 일정·분기 매출로 이를 대신하고 있다. 또한 같은 상권의 변화를 비교하기 위해 고정효과를 제거하면 사용할 공연 활동 변동이 제한되고, 작은 외식업 집단은 매출이 자주 제공되지 않는다.\n\n이 진단은 **자료에서 확인된 측정·표본 구조**와 **비유의성을 설명할 수 있는 가설**을 구분한다. 각각이 비유의성에 기여한 비율이나 진짜 효과의 크기는 현재 자료로 분해할 수 없다. 통계적 유의성보다 먼저 무엇을 측정하는 연구인지 정하고, 그 질문에 맞는 자료를 확보해야 한다.')
md('target','## 1. 관람객의 추가 지출과 지금의 회귀계수는 다르다\n\n관심 있는 경로는 공연 회차·관객 유입 → 인근 식당 방문 → 추가 소비다. 현재 기본 모형은 **등록 공연 ID 수가 달라진 분기에, 같은 상권·업종의 총매출 또는 21~24시 매출 비중이 어떻게 달라지는지** 추정한다. 장기 공연 1편과 하루 공연 1편을 모두 ID 1개로 세므로 관객 노출량과 같지 않다.\n\n시간창 모형은 예정 시작시각과 러닝타임으로 추정한 종료시각을 사용한다. 결과변수는 여전히 분기별 6개 고정 시간대 매출이며, 상권×업종×분기 고정효과를 추가했다. 따라서 **분기 전체의 매출 증가가 아니라 같은 분기 안의 시간대 상대 패턴**을 추정한다. 실제 종료 후 3시간 매출을 관측한 결과가 아니다.')
md('multiplicity','## 2. 다중검정 보정만 때문에 사라진 결과는 아니다\n\n주분석에는 보정 전 5% 기준을 통과한 야간 비중 결과가 2개 있었지만, 시간창의 업종별 40개 결과는 보정 전에도 하나도 통과하지 못했다. 따라서 보정을 없애는 것으로 전체 문제를 설명할 수 없다. p값은 효과의 크기나 연구의 중요도를 나타내지 않는다. [미국통계학회 설명](https://www.amstat.org/asa/files/pdfs/p-valuestatement.pdf)에서도 효과 추정과 불확실성을 함께 해석하도록 권고한다.')
table('families','같은 검정군에서 보정 전후를 비교',[('대상','검정군','text'),('검정수','검정 수','number'),('보정전_5pct','보정 전 p<0.05','number'),('보정후_5pct','보정 후 p<0.05','number'),('보정','방법','text')],'대상')


In [6]:
md('precision_heading','## 3. 일부 추정치는 작은 양의 관계와 음의 관계를 모두 허용한다\n\n다음은 **1+공연 ID 수가 2배**인 모형상 비교다. 공연 수 자체를 정확히 2배로 늘리는 비교와는 다르다. 총매출은 (매출+1)의 조건부 기하평균 상대차이 %, 야간 비중은 %p다. 모든 구간은 개별 95% 신뢰구간이며 실제 원화 매출의 평균 인과효과가 아니다. 점추정치가 0 근처인지, 불확실성이 큰지를 함께 읽어야 한다.')
table('precision','업종별 효과 크기와 개별 95% 신뢰구간',[('industry','업종','text'),('log_sales','총매출 상대차이 % [CI]','text'),('night_share_pct','야간 비중 %p [CI]','text')],'industry')
md('within_heading',f"## 4. 같은 상권 안에서 쓸 수 있는 공연 활동 변동은 제한적이다\n\n주모형은 업종별 상권 고유 특성과 공통 분기 변화를 제거한다. 이에 따라 원래 ln(1+공연 ID 수)의 변동 중 잔차에 남은 제곱합 비율은 업종별 **{variance.fe_variance_retained_pct.min():.2f}~{variance.fe_variance_retained_pct.max():.2f}%**다. 나머지가 모두 오류나 쓸모없는 자료였다는 뜻은 아니다. 상권 간 차이와 공통 분기 변화를 그대로 공연의 효과로 해석하지 않기 위해 제외한 것이다.\n\n다만 대부분 상권에는 실제 시간 변화가 있다. 낮은 잔차 변동 비율만으로 검정력 부족을 확정하거나 상권 대부분에 변화가 없다고 말할 수 없다. 고정효과를 없애 겉보기 관계를 키우면 상권 규모·관광 수요 등 고정된 차이가 다시 섞인다. 필요한 개선은 같은 상권 안에서 실제 관객 노출이 어떻게 바뀌는지를 더 정확히 관측하는 것이다.",'variation')
chart('variation','고정효과 제거 후 남은 공연 활동 변동','industry','fe_variance_retained_pct','잔차 제곱합 / 원시 중심 제곱합 (%)','2023~2025년 · 업종별 주모형 표본 · ln(1+등록 공연 ID 수)')
md('precision_examples','한식 총매출은 −0.43% [−5.08%, +4.45%]로 점추정치가 0에 가깝다. 양식은 +5.71% [−0.94%, +12.82%]로 양의 관계가 추정되지만 불확실성이 크다. 같은 비유의 결과에도 두 상황이 함께 있다. 위 범위에서 한식의 작은 양의 관계를 배제할 수 없고, 양식의 상당한 양의 관계도 배제할 수 없다.','precision')
md('concentration','행 수보다 정보의 분포도 중요하다. 중식은 상위 5개 상권이 잔차화 노출 제곱합의 약 53.6%, 패스트푸드는 약 52.3%를 차지한다. 이 지표는 계수 추정에 쓰이는 설명변수 변동의 집중도이며, 실제 관객이나 매출의 점유율이 아니다. 역 Herfindahl을 “진짜 표본수” 또는 검정 자유도로 바꾸어 쓰지 않는다.','information')
table('information','추정에 쓰이는 변동이 얼마나 여러 상권에 분산되어 있는가',[('industry','업종','text'),('area_clusters','관측 상권','number'),('top5_pct','상위 5개 정보 비중(%)','number'),('inverse_herfindahl_information','정보분포 역 Herfindahl','number')],'industry')
md('coverage_heading',f"## 5. 매출 표본은 작은 집단과 업종을 고르게 포함하지 않는다\n\n전체 격자 {sample['grid_rows']:,}행 중 매출이 관측된 것은 {sample['sales_rows']:,}행({sample['grid_sales_coverage_pct']:.1f}%)이다. 전체 격자에는 점포가 없거나 점포 자료가 없는 집단도 있으므로, 이 수치만으로 모든 외식업 매출의 {100-sample['grid_sales_coverage_pct']:.1f}%가 누락됐다고 말할 수 없다. **전체점포 수가 양수인 집단**으로 분모를 좁혀도 매출 제공률은 {sample['positive_store_sales_coverage_pct']:.1f}%다.")
chart('coverage','점포가 있는 집단에서도 업종별 매출 제공률이 다르다','industry','positive_store_coverage_pct','매출 제공률 (%)','2023~2025년 · 전체점포 수 >0인 상권×업종×분기 · 미제공은 0원으로 채우지 않음')
md('coverage_interpretation',f"같은 상권·분기에 10종 매출이 모두 있는 주모형 표본은 **{sample['all10_main_areas']}개 상권, {sample['all10_main_area_quarters']}개 상권·분기, {sample['all10_main_rows']:,}행**이다. 주모형 {sample['main_model_rows']:,}행의 {sample['all10_main_share_pct']:.1f}%다. 업종별 계수 차이에는 서로 다른 관측 상권의 차이도 섞일 수 있다. 누락된 매출을 모르므로 이 편중이 계수를 0 쪽으로 얼마나 밀었는지, 반대로 부풀렸는지는 확인할 수 없다.\n\n시간창 표본의 {sample['window90_timebin_rows']:,}행은 같은 상권·업종·분기 {sample['window90_base_area_industry_quarters']:,}개를 6개 시간대로 나눈 것이다. 주모형보다 독립적인 지역·기간 정보가 4배 이상 생긴 것으로 해석하면 안 된다.")


In [7]:
qchange=exposure['quarter_change_comparison']
spatial=exposure['spatial']
time_alignment={r['metric']:r for r in exposure['time_alignment']}
sharing=next(r for r in exposure['sharing'] if r['unit']=='performance_ID')
md('exposure_heading','## 6. 작품 수·예정 회차·관객 수는 서로 다른 노출이다')
md('id_difference',f"공연 ID 수와 추정 예정회차 수는 상권·분기 수준에서는 Pearson 상관이 0.788이지만, 인접 분기의 변화량 상관은 0.599다. **{qchange['adjacent_pairs']:,}개 인접 분기 쌍 중 {qchange['opposite_sign_changes']:,}개({100*qchange['opposite_sign_changes']/qchange['adjacent_pairs']:.1f}%)는 변화 방향이 반대**였다.\n\n예정회차는 해석 가능한 일정만으로 재구성했으므로 차이에는 지표 정의와 일정 가용률의 영향이 함께 포함된다. 예정회차를 실제 회차의 정답으로 두고 ID 지표의 오차율을 추정한 것은 아니다. 현재 입력에는 실제 회차·실제 종료시각·관객 수·좌석 점유율이 없다. 시설 좌석 수를 관객 수로 대체하면 빈 좌석과 복수 공연장, 과거 수용 규모를 구분하지 못한다.",'exposure_summary')
md('time_aggregation',f"예정 종료 후 3시간 창 **{time_alignment['post03_spans_multiple_fixed_timebins']['pct']:.1f}%**가 고정 시간 구간 두 개 이상에 걸친다. 그러나 매출 한 행은 해당 분기의 같은 시간 구간을 모두 합한 **{exposure['quarter_timebin_calendar_hours_min']}~{exposure['quarter_timebin_calendar_hours_max']}시간**의 소비다. 20시 30분 공연 종료 후 23시 30분까지의 소비와, 분기 전체 17~21시·21~24시 소비는 같은 관측이 아니다.\n\n짧은 공연 관련 소비가 기타 소비와 섞이는 구조는 확인된다. 그 결과 신호가 얼마나 약해졌는지는 일별 정밀 매출과 실제 공연 이력이 없어서 추정할 수 없다. 공연 창에 겹치는 분 단위 비율로 분기 매출을 나누어도 실제 공연 직후 매출이 생성되지는 않는다.",'exposure_summary')
md('spatial_heading','## 7. “상권 경계까지 500m”는 “그 상권의 모든 식당이 500m 이내”가 아니다')
md('spatial_detail',f"현재 연결은 공연장 점에서 상권 폴리곤까지의 **최단거리**를 사용한다. 활성 시설·상권 {spatial['active_facility_area_pairs']:,}쌍 중 {spatial['pct_pairs_less_than_half_polygon_area_within_500m']:.1f}%는 그 공연장의 500m 원 안에 포함되는 상권 면적이 절반 미만이다. {spatial['pct_pairs_some_polygon_vertices_beyond_1000m']:.1f}%는 상권의 일부 꼭짓점이 그 공연장에서 1km보다 멀다.\n\n이는 시설·상권 쌍별 기하학적 진단이다. 음식점 절반이 멀리 있다는 뜻이나 모든 공연장에서 멀다는 뜻은 아니며, 면적 비중을 매출 비중으로 바꾸어 쓸 수 없다. 다만 상권 전체 매출에는 그 공연장과 거리가 먼 소비가 함께 포함될 수 있다.",'exposure_summary')
md('shared_exposure',f"공연 ID의 {sharing['shared_unit_pct']:.2f}%가 둘 이상의 상권에 연결되며, 평균 {sharing['mean_areas_per_unit']:.2f}개 상권이 같은 공연을 공유한다. 따라서 상권 수를 곧바로 서로 독립적인 공연 충격의 개수로 볼 수 없다.\n\n공유 노출만으로 잔차 공간 상관이 입증되는 것은 아니다. 공간 상관을 무시한 표준오차는 경우에 따라 오히려 너무 작을 수도 있으므로, 이를 큰 p값의 확인된 원인이라고 설명하지 않는다. 좁은 매출 관측 영역과 공연 노출 중첩을 명확히 정의할 필요가 있다는 설계 진단이다.",'exposure_summary')
md('ratio_heading',f"## 8. 야간 매출액·야간 비중·시간대 상대 패턴은 구분해야 한다\n\n같은 상권·업종의 매출을 연속해 관측한 {sample['adjacent_observed_pairs']:,}쌍 중 **{sample['opposite_night_amount_share_direction_pct']:.1f}%는 야간 매출액과 야간 비중의 변화 방향이 반대**였다. 그중 {sample['night_amount_increase_share_decrease_pairs']:,}쌍은 야간 매출액이 증가하면서 비중은 감소했다. 이는 공연 효과를 추정한 결과가 아니라 결과변수의 차이를 보여 주는 기술통계다.\n\n야간 매출이 10% 늘어도 총매출이 20% 늘면 야간 비중은 내려간다. 이 예시는 비율의 성질을 설명하기 위한 가정이다. 연구 질문이 매출 증대라면 비중은 보조 결과로 두고 총매출·관심 시간대 매출액을 각각 정의해야 한다.",'sample_summary')
md('common_heading','시간창의 분기 고정효과는 해당 상권·업종·분기의 공통 로그매출 수준을 제거한다. 이는 총매출을 고정한 비중 모형과도 다르다. 모든 시간대에 비슷하게 나타나는 변화는 흡수되지만, 추가 고정효과가 남은 노출 정보를 전부 없앤 것은 아니다. 종료 후 0~3시간 업종 모형에서는 2종 고정효과만 사용했을 때의 잔차 제곱합 중 약 62.6~68.0%가 3종 고정효과 후에도 남는다.','variation')
table('time_common','시간창 공통 기울기: 1+노출 2배 비교',[('label','예정 시간창','text'),('estimate_ci','상대차이 % [개별95% CI]','text'),('raw_p','보정 전 p','text'),('adjusted_p','Holm p','text')],'label')
md('common_read','종료 후 0~3시간 공통 관계는 −2.75% [−7.00%, +1.68%]다. 공통 기울기 가정 아래 큰 일률적 양의 상대 패턴은 이 구간과 양립하기 어렵다. 그러나 업종별 구간은 훨씬 넓고, 이 계수는 총매출 성장률도 실제 관람객의 인과효과도 아니다. 따라서 “무조건 표본을 늘리면 양의 유의성이 나올 것”이라고 기대할 근거도 없다.','time_common')


In [8]:
md('causes','## 9. 확인한 제약과 아직 확인하지 못한 원인을 구분한다\n\n**확인한 구조:** 시간·공간 집계가 연구 질문보다 넓고, 공연 ID와 예정회차의 변화가 다르며, 실제 관객은 미관측이다. 고정효과 후 변동이 줄고 일부 업종의 정보가 소수 상권에 집중된다. 미제공 매출로 인해 업종별 표본이 달라진다.\n\n**아직 검증하지 못한 가설:** 실제 추가 소비가 작거나, 관람객이 다른 지역에서 소비하거나, 기존 소비자의 방문 감소가 관람객 소비를 상쇄할 수 있다. 공연 전 식사와 공연 후 음료처럼 업종·시간에 따라 반응이 다를 수도 있다. 이러한 경로는 경제적으로 가능하지만 현재 회귀의 비유의성을 설명하는 확정 원인이 아니다.\n\n**확인되지 않은 설명:** 데이터 수집 실패나 회귀 계산 오류가 이번 결과를 만들었다는 근거는 찾지 못했다. 64개 기존 계수·표준오차·구간·p값을 같은 표본으로 재현했고 최대 차이는 약 2.22×10⁻¹⁶이었다. 재현성은 계산 검증이며 누락된 공연·과거 일정의 정확성·인과성까지 증명하지 않는다.')
md('next_steps','## 10. 다음 분석은 무엇부터 바꿀 것인가\n\n1. **결과변수와 질문을 먼저 고정한다.** 목표가 소비 증가라면 총매출·관심 시간대 매출액을 중심으로 정의하고 야간 비중은 보조 지표로 둔다. 일별 정밀 자료가 없을 때의 결론은 분기 관련성으로 한정한다.\n2. **실제 공연 노출을 검증한다.** 예정회차·시설 좌석으로 관객을 단정하지 않는다. 확인 가능한 공연의 실제 날짜·회차·관객 또는 티켓 수를 확보하고, 특히 장기 공연과 휴일 일정의 과거 이력을 대조한다.\n3. **관측 공간과 비교일을 맞춘다.** 작은 격자나 점포별 매출을 확보할 수 있다면 공연장 가까운 소비와 먼 소비를 구분한다. 공연일과 같은 요일의 비교일, 공휴일·날씨·다른 행사와 주변 공연 중첩을 설계에 반영한다. 일별 자료만 확보돼도 공연일 구분은 개선되지만 6구간 자료로 정확한 종료 후 3시간을 복원할 수는 없다.\n4. **자료 확보 후 검출 가능한 크기를 사전에 평가한다.** 논문에서 의미 있게 볼 변화와 주 분석 시간창·업종·표본을 정하고 모의실험 또는 설계 기반 검정력 계산으로 규모를 결정한다. 현재 SE를 사용한 대략적 검출규모는 보조 CSV로 남겼으며, 효과가 실제 존재할 확률이나 다음 자료의 검정력으로 읽지 않는다.\n\n지금 가장 우선순위가 높은 일은 모형의 복잡도를 높이는 것이 아니라 **짧고 국소적인 소비 반응을 관측할 수 있는 자료와 실제 공연 노출을 맞추는 것**이다. 자료를 개선해도 유의성이 보장되지는 않는다. 충분히 좁은 구간이 0 주변에 남는다면 작은 관계 자체가 연구 결과가 될 수 있다.')
md('model_details','## 분석 정의와 재현 범위\n\n기본 모형: Yₐᵢq = αₐᵢ + λᵢq + βᵢ ln(1+등록ID수ₐq) + εₐᵢq. Y는 ln(1+분기 총매출) 또는 21~24시 매출/총매출×100이다.\n\n시간창 모형: Zₐᵢqb = αₐᵢb + λᵢqb + γₐᵢq + βᵢ ln(1+예정창 노출ₐqb) + εₐᵢqb. Z는 ln(1+분기 시간대 매출), b는 6개 고정 시간대다. 예정창 노출은 회차별 창이 시간대와 겹치는 분을 합해 180으로 나눈 값이며, 창끼리 중첩되면 각각 합산한다. 완전한 실제 노출이 아니라 해석 가능한 일정의 합계다.\n\n표준오차는 상권 군집 기준이다. 진단은 기존 주모형·시간창 모형을 유지했으며 거리·업종·시간창을 p값에 따라 새로 고르지 않았다. 매출 미제공은 0원으로 대체하지 않았다. 남은 질문은 실제 관객량·정밀 매출·공간 중첩을 관측했을 때 추정 범위가 얼마나 좁아지는가이며, 현재 자료만으로 답할 수 없다.')


## 3. 단일 HTML 저장 및 검증

보고서와 차트는 검증된 데이터와 함께 자체 포함 HTML로 만든다. 본문·표·차트와 원본 데이터 경로를 보존하고 보고서 입력 해시가 바뀌지 않았는지 검사한다.

In [9]:
generated=datetime.now(ZoneInfo('Asia/Seoul')).isoformat()
artifact={'surface':'report','manifest':{'version':1,'surface':'report','title':title,
 'generatedAt':generated,'description':'비유의성을 측정·식별변동·표본·정밀도로 나누어 검토한 졸업 프로젝트 진단',
 'blocks':blocks,'tables':tables,'charts':charts,'sources':sources},
 'snapshot':{'version':1,'generatedAt':generated,'status':'ready','datasets':datasets,'accessIssues':[]},'sources':sources}
assert all(sha(ROOT/path)==value for path,value in input_hashes.items())
(OUT/'artifact.json').write_text(json.dumps(artifact,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
notes={'audience':'technical','delivery':'portable_html','question':'비유의성의 확인된 구조와 미확정 원인',
 'source_sha256':input_hashes,'existing_models_modified':False,'specification_search':False,
 'required_structure_mapping':{'title':'title','technical_summary':'summary','findings':'multiplicity/precision_heading/within_heading/coverage_heading/exposure_heading/spatial_heading/ratio_heading',
 'definitions':'target','methodology':'model_details','limitations':'causes','next_steps':'next_steps','further_questions':'model_details'},
 'chart_map':[{'id':'variation','type':'horizontalBar','rows':10,'claim':'FE 후 노출 변동 보존율의 업종 차이'},
              {'id':'coverage','type':'horizontalBar','rows':10,'claim':'점포가 있는 집단의 매출 제공률 차이'}],
 'table_rationale':'개별95%CI와 단위, 보정 전후 p값, 정보집중도 등의 정확한 조회',
 'omitted_charts':'원인별 기여율은 식별되지 않아 원인 기여도 차트는 작성하지 않음',
 'sharing_assessment':'Share with caveats'}
(OUT/'report_notes.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')
print('본문',len(blocks),'표',len(tables),'차트',len(charts))

본문 28 표 4 차트 2


In [10]:
builder=Path('C:/Users/mamap/.codex/plugins/cache/openai-curated-remote/data-analytics/0.2.10-13ceeea1f599/skills/build-report/scripts/deliver_portable_artifact.mjs')
result=subprocess.run([shutil.which('node'),str(builder),'--input',str(OUT/'artifact.json'),'--output',str(READABLE/'report.html')],
                      capture_output=True,text=True,encoding='utf-8',timeout=120)
print(result.stdout[-6000:])
if result.returncode: print(result.stderr[-6000:])
assert result.returncode==0
(OUT/'delivery_receipt.json').write_text(result.stdout,encoding='utf-8')
receipt=json.loads(result.stdout.strip())
notes['delivery_verification']=receipt['stages']['verification']
notes['browser_visual_qa_performed']=receipt['stages']['verification']=='passed'
notes['qa_limitation']='브라우저 실행 환경이 없어 HTML 화면 자동검사는 수행하지 않았음. 데이터·원문 페이로드·문서 구조 검증은 통과.' if not notes['browser_visual_qa_performed'] else ''
(OUT/'report_notes.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')


{"ok":true,"html":"C:\\Coding\\졸업프로젝트\\reports\\main\\null_diagnostics\\report.html","stages":{"validation":"passed","package":"passed","verification":"structural_only"},"browserWarning":{"code":"browser_unavailable","message":"No installed Chromium headless-shell executable was found. Set CHROMIUM_EXECUTABLE_PATH (or the legacy PLAYWRIGHT_EXECUTABLE_PATH) or preinstall Chromium headless-shell under PLAYWRIGHT_BROWSERS_PATH; portable tooling never downloads browsers."},"counts":{"blocks":28,"charts":2,"html":0,"metrics":0,"tables":4},"sourceDialog":"not_verified","sourceInteraction":"not_verified","viewports":[],"timings":{"validateAndPackageMs":129.6,"staticChartExtractionMs":53,"structuralVerificationMs":4.8,"totalMs":142.3}}



2884